<h2 align='center'>Flight_Price_Prediction_Model ML Flow</h2>

In [1]:
# Import Libraries
import pandas as pd
import numpy as np

from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

from joblib import dump, load

import warnings
warnings.filterwarnings("ignore")

In [3]:
# Load Dataset
flights_dataset = pd.read_csv("C:/Users/User/Productionization of ML Systems/travel_capstone/flights.csv", parse_dates=['date'])
flights_dataset.drop(['travelCode', 'userCode'], axis=1, inplace=True)

In [4]:
# one hot encoding categorical variables
ohe = OneHotEncoder(drop='first')
ohe.fit(flights_dataset[['from', 'to', 'flightType', 'agency']])
encoded_var = ohe.transform(flights_dataset[['from', 'to', 'flightType', 'agency']])
encoded_df = pd.DataFrame(encoded_var.toarray(), columns=ohe.get_feature_names_out(), dtype=int)

In [5]:
# merge encoded_variables with main dataframe
flights_dataset = pd.concat([flights_dataset,encoded_df], axis=1)

# drop encoded_variables original columns from main dataframe
flights_dataset.drop(columns=['from', 'to', 'flightType', 'agency'], inplace=True)

In [6]:
# Separate features and target variable
X = flights_dataset.drop(['date','price'], axis=1)
y = flights_dataset['price']

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [7]:
# ML Model - 2 Implementation
# Initialize Random Forest Regressor
rf_model = RandomForestRegressor(random_state=42)

# Fit the Algorithm
rf_model.fit(X_train, y_train)

# Predict on the model
y_pred_rf = rf_model.predict(X_test)

In [8]:
# Evaluation Metrics
mse_rf = mean_squared_error(y_test, y_pred_rf)
mae_rf = mean_absolute_error(y_test, y_pred_rf)
r2_rf = r2_score(y_test, y_pred_rf)

# evaluation Metric Score
print("Random Forest Model Evaluation Metrics:")
print("Mean Squared Error (MSE):", mse_rf)
print("Mean Absolute Error (MAE):", mae_rf)
print("R-squared (R2) Score:", r2_rf)

# dump(rf_model, 'rf_model.joblib') # save the rf_model model
# dump(ohe, 'cat_encoder.joblib') # save the categorical encoder

Random Forest Model Evaluation Metrics:
Mean Squared Error (MSE): 2.3914503196758745e-23
Mean Absolute Error (MAE): 3.60094037211661e-12
R-squared (R2) Score: 1.0


### Track Experiments

In [10]:
pip install mlflow dagshub

  Using cached pyasn1_modules-0.4.2-py3-none-any.whl.metadata (3.5 kB)
  Using cached rsa-4.9.1-py3-none-any.whl.metadata (5.6 kB)
   ---------------------------------------- 0.0/10.2 MB ? eta -:--:--
   ------- -------------------------------- 1.8/10.2 MB 12.3 MB/s eta 0:00:01
   ----------------- ---------------------- 4.5/10.2 MB 12.7 MB/s eta 0:00:01
   --------------------- ------------------ 5.5/10.2 MB 10.9 MB/s eta 0:00:01
   ------------------------- -------------- 6.6/10.2 MB 8.8 MB/s eta 0:00:01
   --------------------------- ------------ 7.1/10.2 MB 7.8 MB/s eta 0:00:01
   ---------------------------- ----------- 7.3/10.2 MB 7.1 MB/s eta 0:00:01
   ----------------------------- ---------- 7.6/10.2 MB 6.2 MB/s eta 0:00:01
   ------------------------------- -------- 8.1/10.2 MB 5.0 MB/s eta 0:00:01
   --------------------------------- ------ 8.4/10.2 MB 4.6 MB/s eta 0:00:01
   ---------------------------------- ----- 8.7/10.2 MB 4.3 MB/s eta 0:00:01
   -----------------------


[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [18]:
import mlflow
import mlflow.sklearn

# dagshub setup
import dagshub
dagshub.init(repo_owner='narasimman2k', repo_name='Flight_Price_Prediction_Model', mlflow=True)

Initialized MLflow to track repo "narasimman2k/Flight_Price_Prediction_Model"

Repository narasimman2k/Flight_Price_Prediction_Model initialized!

In [19]:
# Models list with details for tracking
models = [
    (
        "Decision Tree Regressor",
        {"random_state": 45},
        DecisionTreeRegressor(random_state=45),
    ),
    (
        "Random Forest Regressor",
        {"random_state": 45},
        RandomForestRegressor(random_state=45),
    ),
    (
        "Gradient Boosting Regressor",
        {"random_state": 45},
        GradientBoostingRegressor(random_state=45),
    ),
    (
        "Linear Regression",
        {},
        LinearRegression(),
    )
]

In [20]:
# Initialize MLflow
mlflow.set_experiment("Regression Model Evaluation")

for model_name, params, model in models:
    # Train the model
    model.set_params(**params)
    model.fit(X_train, y_train)
    
    # Predict and evaluate
    y_pred = model.predict(X_test)
    mse = mean_squared_error(y_test, y_pred)
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)

    print("\n\n")
    print(f"{model_name} Evaluation Metrics:")
    print(f"Mean Squared Error (MSE): {mse}")
    print(f"Mean Absolute Error (MAE): {mae}")
    print(f"R-squared (R2) Score: {r2}\n")

    # Log results in MLflow
    with mlflow.start_run(run_name=model_name):
        mlflow.log_params(params)
        mlflow.log_metrics({
            "mean_squared_error": mse,
            "mean_absolute_error": mae,
            "r2_score": r2
        })

        # Log the model
        mlflow.sklearn.log_model(model, "model")


2026/03/09 16:26:52 INFO mlflow.tracking.fluent: Experiment with name 'Regression Model Evaluation' does not exist. Creating a new experiment.





Decision Tree Regressor Evaluation Metrics:
Mean Squared Error (MSE): 8.736174426338578e-23
Mean Absolute Error (MAE): 7.143983624129047e-12
R-squared (R2) Score: 1.0



2026/03/09 16:26:55 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/09 16:27:00 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run Decision Tree Regressor at: https://dagshub.com/narasimman2k/Flight_Price_Prediction_Model.mlflow/#/experiments/0/runs/9ff78dacb8a94259aef8361ff4948fdb
🧪 View experiment at: https://dagshub.com/narasimman2k/Flight_Price_Prediction_Model.mlflow/#/experiments/0



Random Forest Regressor Evaluation Metrics:
Mean Squared Error (MSE): 2.4034042621401377e-23
Mean Absolute Error (MAE): 3.60975257596737e-12
R-squared (R2) Score: 1.0



2026/03/09 16:28:15 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/09 16:28:19 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run Random Forest Regressor at: https://dagshub.com/narasimman2k/Flight_Price_Prediction_Model.mlflow/#/experiments/0/runs/e2960bff86ea461b95594693d4a09243
🧪 View experiment at: https://dagshub.com/narasimman2k/Flight_Price_Prediction_Model.mlflow/#/experiments/0



Gradient Boosting Regressor Evaluation Metrics:
Mean Squared Error (MSE): 1921.9072285971536
Mean Absolute Error (MAE): 34.684251765852345
R-squared (R2) Score: 0.9854145948077329



2026/03/09 16:29:16 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/09 16:29:20 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run Gradient Boosting Regressor at: https://dagshub.com/narasimman2k/Flight_Price_Prediction_Model.mlflow/#/experiments/0/runs/2512b09b16f84723bbc0b31464225189
🧪 View experiment at: https://dagshub.com/narasimman2k/Flight_Price_Prediction_Model.mlflow/#/experiments/0



Linear Regression Evaluation Metrics:
Mean Squared Error (MSE): 10640.311874028786
Mean Absolute Error (MAE): 81.32138381774065
R-squared (R2) Score: 0.9192503895372307



2026/03/09 16:29:36 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/09 16:29:40 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run Linear Regression at: https://dagshub.com/narasimman2k/Flight_Price_Prediction_Model.mlflow/#/experiments/0/runs/893cdd9e74c94b85baa660f68ea7baf8
🧪 View experiment at: https://dagshub.com/narasimman2k/Flight_Price_Prediction_Model.mlflow/#/experiments/0


View ML Flow experiment at: https://dagshub.com/narasimman2k/Flight_Price_Prediction_Model.mlflow